# Example: Breast Cancer Diagnosis

This demonstration uses the **GradFlow** engine to solve a **Breast Cancer Diagnosis** classification problem.

### 1. Imports and environment setup

In [2]:
import random
from gradflow.utils import read_csv
from gradflow.tensor import Tensor
from gradflow.nn.mlp import MLP

random.seed(1337)

### 2. Dataset Preparation
We use our custom `read_csv` utility to load the clinical data. The labels are `1.0` (Malignant) and `-1.0` (Benign).

In [3]:
data = read_csv('medical_data.csv')
print(f"Loaded {len(data)} clinical samples.")

raw_features = []
labels = []

for row in data:
    raw_features.append([
        float(row['radius_mean']), 
        float(row['texture_mean']), 
        float(row['smoothness_mean'])
    ])
    labels.append(float(row['diagnosis']))

# Z-score Normalization
import math
n_features = len(raw_features[0])
means = [sum(f[i] for f in raw_features) / len(raw_features) for i in range(n_features)]
stds = [math.sqrt(sum((f[i] - means[i])**2 for f in raw_features) / len(raw_features)) for i in range(n_features)]

features = []
for f in raw_features:
    normalized = [(f[i] - means[i]) / (stds[i] if stds[i] > 0 else 1.0) for i in range(n_features)]
    features.append(normalized)

print("Features normalized using Z-score scaling.")

Loaded 40 clinical samples.
Features normalized using Z-score scaling.


### 3. Model Architecture
Our model uses Gaussian initialization to ensure neurons start in an active state.

In [4]:
model = MLP(3, [6, 6, 1])
print(f"Network initialized with {len(model.parameters())} parameter Tensors.")

Network initialized with 26 parameter Tensors.


### 4. Training Engine
We define our Batch Gradient Descent update rule here.

In [5]:
def update_weights(model_parameters, learning_rate):
    for p in model_parameters:
        def update(item):
            if isinstance(item, list):
                for x in item: update(x)
            else: # Value object
                item.data -= learning_rate * item.grad
        update(p.data)

### 5. Training Loop
We train for 500 epochs with a learning rate of 0.01.

In [6]:
epochs = 500
learning_rate = 0.05

for i in range(epochs):
    predictions = [model(x) for x in features]
    # prediction.data[0] is the scalar Value output of the linear unit
    loss = sum((prediction.data[0] - target_label)**2 for target_label, prediction in zip(labels, predictions)) * (1.0 / len(labels))
    
    model.zero_grad()
    loss.backward()
    
    update_weights(model.parameters(), learning_rate)
    
    if i % 100 == 0 or i == epochs - 1:
        print(f"Epoch {i:3d} | Loss: {loss.data:.4f}")

Epoch   0 | Loss: 1.4076
Epoch 100 | Loss: 0.0481
Epoch 200 | Loss: 0.0325
Epoch 300 | Loss: 0.0225
Epoch 400 | Loss: 0.0158
Epoch 499 | Loss: 0.0116


### 6. Clinical Inference & Evaluation
We evaluate the model on the full dataset to compute the Confusion Matrix metrics (True/False Positives and Negatives) alongside standard accuracy, precision, and recall.

In [8]:
tp = tn = fp = fn = 0

for x, y_true in zip(features, labels):
    out = model(x).data[0].data
    y_pred = 1.0 if out > 0 else -1.0
    
    if y_pred == 1.0 and y_true == 1.0:
        tp += 1
    elif y_pred == -1.0 and y_true == -1.0:
        tn += 1
    elif y_pred == 1.0 and y_true == -1.0:
        fp += 1
    elif y_pred == -1.0 and y_true == 1.0:
        fn += 1

accuracy = (tp + tn) / len(labels)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")
print(f"\nConfusion Matrix:")
print(f"          Predicted: ")
print(f"          +1 (M) | -1 (B)")
print(f"Actual +1 | {tp:6d} | {fn:6d}")
print(f"Actual -1 | {fp:6d} | {tn:6d}")

Accuracy:  100.00%
Precision: 100.00%
Recall:    100.00%
F1-Score:  100.00%

Confusion Matrix:
          Predicted: 
          +1 (M) | -1 (B)
Actual +1 |     36 |      0
Actual -1 |      0 |      4
